# Multi-LoRA email workshop: train four adapters

**Primary question:** Can independently trained finance, travel, and operations adapters specialize one shared base without training on the test set?

**Time:** 30 minutes instruction plus hardware-dependent training. **Base:** selected through a pinned model profile in the first code cell. The default is Qwen2.5 0.5B for accessible local iteration; Nemotron Nano 8B remains an optional NVIDIA profile with tighter memory and prompt-contract requirements.

Prerequisite: complete [01: data and routing](01_email_data_and_routing.ipynb).

## Training contract

- Train **generalist**, **operations**, **finance**, and **travel** independently from the same base revision and seed.
- Reinitialize the base/LoRA/optimizer for each adapter; never continue the prior specialist's learned weights.
- Freeze base weights; rank-8 LoRA targets the projection modules declared by the selected model profile.
- A custom model must provide an immutable revision, compatible chat template, EOS token, and explicit LoRA target-module names.
- Apply loss only to the JSON completion and EOS. Prompt tokens and padding use `-100` labels.
- Preserve the complete target when shortening long email bodies; never truncate the answer.
- Select the best checkpoint by validation loss. The test partition is never passed to Trainer.
- Save local safetensors, tokenizer, manifest, checksums, and local Trackio metrics. No Hub push or cloud job.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

import pandas as pd
from IPython.display import display

import email_workshop
from email_workshop.data import ADAPTERS, load_splits, read_json
from email_workshop.model_profiles import DEFAULT_MODEL_PROFILE, model_profile_keys, resolve_model_profile
from email_workshop.training import TrainConfig, train_adapter, verify_adapters

WORKSHOP_ROOT = Path(email_workshop.__file__).resolve().parents[1]
ARTIFACT_DIR = WORKSHOP_ROOT / Path(os.getenv('EMAIL_WORKSHOP_RUN', 'artifacts/run-2026'))
# Choose any key printed by model_profile_keys(), or use 'custom'.
MODEL_CHOICE = DEFAULT_MODEL_PROFILE
CUSTOM_MODEL_ID = None
CUSTOM_MODEL_REVISION = None
CUSTOM_TARGET_MODULES = None
MODEL = resolve_model_profile(
    MODEL_CHOICE,
    model_id=CUSTOM_MODEL_ID,
    revision=CUSTOM_MODEL_REVISION,
    target_modules=tuple(CUSTOM_TARGET_MODULES) if CUSTOM_TARGET_MODULES else None,
)
RUN_TOKENIZATION_CHECK = False
RUN_TRAINING = False
SMOKE_ONLY = True
DEVICE = 'cuda'
ADAPTERS_TO_TRAIN = ADAPTERS
CONFIG = TrainConfig(
    model_profile=MODEL.key,
    model_id=MODEL.model_id,
    model_revision=MODEL.revision,
    target_modules=MODEL.target_modules,
    device=DEVICE,
    max_steps=2 if SMOKE_ONLY else -1,
    sample_limit=16 if SMOKE_ONLY else None,
)
print(f'Kernel: {sys.executable}')
print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Model choice: {MODEL.key} -> {MODEL.model_id}@{MODEL.revision}')
print(f'LoRA targets: {MODEL.target_modules}; available choices: {model_profile_keys()}')
print(f'Training enabled: {RUN_TRAINING}; smoke only: {SMOKE_ONLY}')
print(f'Available model choices: {model_profile_keys()}')


## Verify the training environment

Install the `[training]` extra from the workshop guide, select this kernel, and restart it after installation. The core notebook environment deliberately does not install a GPU runtime.

A CUDA PyTorch wheel is required for GPU training. CPU is supported for a tiny smoke test, not recommended for the full 0.5B workshop. Training is opt-in. Neither running this notebook as shipped nor the CPU validation notebook submits paid work.

In [ ]:
versions = {}
for package in ['torch', 'transformers', 'peft', 'accelerate', 'trackio']:
    try:
        versions[package] = version(package)
    except PackageNotFoundError:
        versions[package] = 'Not installed; install [training] before enabling training'
display(pd.Series(versions))
if RUN_TRAINING or RUN_TOKENIZATION_CHECK:
    import torch
    print(f'CUDA available: {torch.cuda.is_available()}')
    if DEVICE == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable. See GPU installation instructions.')

## Verify data lineage before spending compute

Use the same run directory across all notebooks. If you change the seed, taxonomy, dataset, base model, or hyperparameters, use a new run directory. Adapter directories are never silently overwritten.

A smoke run produces real trained adapters but insufficient training for quality claims. Marked smoke manifests are rejected by evaluation unless you explicitly allow them. Use a separate artifact root for the later full run.

In [ ]:
splits, data_manifest = load_splits(ARTIFACT_DIR)
display(pd.Series({name: len(rows) for name, rows in splits.items()}))
display(pd.Series(CONFIG.__dict__))
existing = [name for name in ADAPTERS_TO_TRAIN if (ARTIFACT_DIR / 'adapters' / name).exists()]
print('Existing adapter directories:', existing)
if RUN_TRAINING and existing:
    raise FileExistsError('Select only missing adapters or prepare a fresh run root. Existing adapters are preserved.')

## Inspect completion-only training

Turn on `RUN_TOKENIZATION_CHECK` to download the public tokenizer and inspect token counts. The cell prints the expected JSON label array, not email text. Finance/travel examples outside the adapter's scope must produce `[]`.

The training helper shortens body tokens first, keeps system instructions and chat delimiters, and reserves 128 tokens for the complete response. This same prompt budget is used by both inference backends.

In [ ]:
if RUN_TOKENIZATION_CHECK:
    from transformers import AutoTokenizer
    from email_workshop.training import training_example
    tokenizer = AutoTokenizer.from_pretrained(CONFIG.model_id, revision=CONFIG.model_revision)
    rows = []
    for adapter in ADAPTERS:
        example = training_example(tokenizer, splits['train'][0], adapter, CONFIG.max_length)
        target_ids = [token for token in example['labels'] if token != -100]
        rows.append({'adapter': adapter, 'total_tokens': len(example['input_ids']), 'loss_tokens': len(target_ids), 'target': tokenizer.decode(target_ids, skip_special_tokens=True)})
    display(pd.DataFrame(rows))
else:
    print('Tokenization inspection skipped; set RUN_TOKENIZATION_CHECK = True to enable.')

## Run training explicitly

1. For a pipeline smoke test keep `SMOKE_ONLY = True`, choose a fresh run, and set `RUN_TRAINING = True`.
2. Check that all four adapter manifests were produced and local losses are finite.
3. For actual comparison prepare another run, set `SMOKE_ONLY = False`, and run all four adapters with identical settings.
4. Do not tune epochs or prompts against the test partition.

Training includes local Trackio logging. `trackio show --project email-multilora-local` opens its dashboard when you choose to run it. The workshop does not start a dashboard or publish a Space automatically.

In [ ]:
if RUN_TRAINING:
    training_summary = []
    for adapter in ADAPTERS_TO_TRAIN:
        result = train_adapter(ARTIFACT_DIR, adapter, CONFIG)
        training_summary.append({'adapter': adapter, 'purpose': result['purpose'], 'train_rows': result['train_rows'], 'trainable_parameters': result['trainable_parameters'], 'elapsed_seconds': result['elapsed_seconds'], 'train_loss': result['metrics']['train_loss']})
    display(pd.DataFrame(training_summary))
else:
    print('Training skipped. Set RUN_TRAINING = True only after reviewing the configuration.')

## Review losses and saved adapter identities

A lower validation loss does not prove higher classification F1. Finance and travel have many negative examples; an adapter that always emits `[]` can have deceptively low loss. Inspect positive recall in notebook 03.

`verify_adapters` requires all four manifests, the same data/configuration, and matching weight/config checksums before inference.

In [ ]:
completed = [name for name in ADAPTERS if (ARTIFACT_DIR / 'adapters' / name / 'manifest.json').exists()]
if completed:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    for adapter in completed:
        result = read_json(ARTIFACT_DIR / 'adapters' / adapter / 'manifest.json')
        history = pd.DataFrame(result['history'])
        if 'eval_loss' in history:
            valid = history.dropna(subset=['eval_loss'])
            ax.plot(valid['epoch'], valid['eval_loss'], marker='o', label=adapter)
    ax.set(xlabel='Epoch', ylabel='Validation completion loss', title='Adapter training diagnostics')
    ax.legend()
    plt.show()
else:
    print('No completed adapter manifests yet.')
if len(completed) == len(ADAPTERS):
    manifests = verify_adapters(ARTIFACT_DIR)
    display(pd.DataFrame([{'adapter': name, 'purpose': item['purpose'], 'trainable_parameters': item['trainable_parameters'], 'total_parameters': item['total_parameters']} for name, item in manifests.items()]))

## Interpretation checklist

- Can you explain which tokens contribute loss and why padding does not?
- Did every specialist start from the same base, not the previous adapter?
- Does every specialist see off-domain negatives?
- Are all four artifacts present with the same base revision and data fingerprint?
- Is this a smoke or full workshop run?
- Are you keeping the test set untouched?

Continue to [03: compare and serve](03_compare_and_serve.ipynb).